<a href="https://colab.research.google.com/github/yassinmmohey/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassinmmohey/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
%pip -q install duckdb huggingface_hub


In [19]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a
# cell — this repo is public. Use a Colab Secret named HF_TOKEN (key panel, left sidebar).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis (after aggregation):** one row = one **content item within a client** (`client_hash_id`, `content_hash_id`), summarized over calendar **March 2026**.

**Raw source grain:** `fact_content_daily_performance` is `report_date × client_hash_id × content_hash_id` (one row per content item per day it was tracked), partitioned by `month=YYYY-MM`. I roll this up to content-item grain myself with `GROUP BY`.

**Table(s) used:** `fact_content_daily_performance` (`month=2026-03` partition only — the heavy table, so I touch exactly one month of it), joined to `dim_content` (content metadata, grain = one row per content item) and `dim_clients` (history-coverage flags, grain = one row per client).

**Time window:** `report_date` between `2026-03-01` and `2026-03-31` for every daily metric (`gsc_*`, `ga4_*`, `sessions_ai`). `dim_content` fields (`content_type`, `word_count`, `content_created_at`, ...) are not date-windowed — they are as-of-now content properties, so `content_age_days` has to be computed *relative to* the window (age as of 2026-03-31), not to today.

**One thing I deliberately exclude:** `fact_content_query_90d`. Its 90-day window is fixed to the *end* of the snapshot, not to March — joining it onto a March-only slice would silently mix two different time windows (the exact trap the skill warns about: "two tables with the same column name but different windows"). I'll bring it in later, once my feature window and label window are explicitly lined up against it.

In [20]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'  # mid-panel month — NOT the sealed _sample (June 2026)

TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_month':   f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')",
}

# Confirm the real column names before writing a single WHERE clause against them —
# a data contract built on a guessed schema is still a guess.
for name, src in TABLES.items():
    cols = con.sql(f'DESCRIBE SELECT * FROM {src} LIMIT 0').df()['column_name'].tolist()
    print(f'{name}: {cols}\n')


dim_clients: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']

dim_content: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

fact_month: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id` | Context | join/group keys only, never a feature (flyrank-data skill) |
| `report_date` | Context | defines the window, not a model input |
| `keyword_hash_id`, `url_hash_id` (dim_content) | Context | grouping/dedup only |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | Feature | Search Console facts, already observed by month-end, before any refresh decision |
| `ga4_sessions` (where `ga4_data_available IS TRUE`) | Feature | GA4 facts already collected — but only real rows, per the three-valued-flag warning |
| `sessions_ai` | Feature | already-observed AI-referral sessions (sparse — EDA-grade signal per the guide, not a standalone classifier target) |
| `content_type`, `main_intent`, `word_count` (dim_content) | Feature | static content metadata, fixed at publish time |
| `content_created_at` (dim_content) → `content_age_days` | Feature | fixed at publish time, derivable at any decision moment |
| `pct_change_mar` (first-half vs second-half March impressions, built by me) | **Label / proxy** | this is what I threshold to build the proxy target below — it is the trend signal itself, so per the skill's label-trap rule it can NEVER also be a feature |
| `declined_within_march` (proxy label, built by me) | Label / proxy | a **within-month proxy** for the real Lane 2 target. The honest capstone label is a *future* outcome (prior-90d features → next-30d decline); this month-only slice can't build that yet, so I'm explicit that this is a teaching proxy, not the capstone label |
| `fact_content_query_90d` (whole table) | Excluded | window overlaps the snapshot's final months, misaligned with a March-only slice (see §1) |
| `gsc_data_start`, `ga4_data_start`, `access_profile` (dim_clients) | Context | used to check per-client history coverage, not fed to a model |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Fact 1 — grain: one row really is `report_date × client_hash_id × content_hash_id`

In [21]:
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_month']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'duplicate-grain rows found: {len(grain_probe)}  (expect 0)')
grain_probe


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows found: 0  (expect 0)


,report_date,client_hash_id,content_hash_id,c


### Fact 2 — row count and date span of my slice
55 of 104 clients (53%) have March 2026 coverage — confirming the panel is unbalanced, consistent with §4

In [22]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_month']}
""").df()

span


,n_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### Fact 3 — availability: filter with `IS TRUE`, show survival count

NULL vs FALSE both correctly excluded by IS TRUE, but the distinction between them isn't defined in the docs I have; a stronger contract would resolve this before using GA4 fields downstream.

In [23]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_true,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE AND ga4_data_available IS NOT NULL THEN 1 ELSE 0 END) AS ga4_available_false,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_available_null
    FROM {TABLES['fact_month']}
""").df()

availability['pct_survive_is_true'] = (availability['ga4_available_true'] / availability['total_rows'] * 100).round(1)
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_true,ga4_available_false,ga4_available_null,pct_survive_is_true
0,9841378,413966.0,6408671.0,3018741.0,4.2


### Five features (max) — built from `month=2026-03`, rolled up to content-item grain

In [25]:
features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)                                             AS gsc_impressions_mar,
        SUM(f.gsc_clicks)                                                  AS gsc_clicks_mar,
        AVG(f.gsc_avg_position)                                            AS gsc_avg_position_mar,
        SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.ga4_sessions END) AS ga4_sessions_mar,
        -- the leak-in-waiting: first half vs second half of the SAME month, kept OUT of
        -- the feature list on purpose (see the trap below)
        SUM(CASE WHEN f.report_date <  DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_month']} f
    GROUP BY 1, 2
    HAVING imp_first_half > 0
""").df()

content_ctx = con.sql(f"""
    SELECT content_hash_id, content_type, main_intent, word_count,
           DATE_DIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days
    FROM {TABLES['dim_content']}
""").df()

features = features.merge(content_ctx, on='content_hash_id', how='left')
print(f'{len(features):,} content items with March impressions')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151,981 content items with March impressions


,client_hash_id,content_hash_id,gsc_impressions_mar,gsc_clicks_mar,gsc_avg_position_mar,ga4_sessions_mar,imp_first_half,imp_second_half,content_type,main_intent,word_count,content_age_days
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,NaN,57.0,20.0,keyword article,informational,3579,47
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,NaN,199.0,403.0,keyword article,informational,2455,47
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,NaN,467.0,343.0,keyword article,informational,3653,47
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,8.978086,NaN,56.0,26.0,keyword article,informational,3096,47
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,NaN,771.0,1087.0,keyword article,informational,3305,47


**Five features, and why each is knowable at the decision moment:**

1. `gsc_impressions_mar` — knowable because it's Search Console data already logged by the end of March, before an editor ever opens the review queue.
2. `gsc_avg_position_mar` — same: an already-observed ranking average, not a future outcome.
3. `ga4_sessions_mar` — knowable, but only where `ga4_data_available IS TRUE`; I don't invent engagement numbers for rows that were never tracked.
4. `content_age_days` — fixed the moment the page was published (`content_created_at`), so it's knowable at any point afterward, including decision time.
5. `word_count` — a static content property recorded when the article was written; it doesn't change based on what happens next.

`imp_first_half` / `imp_second_half` are deliberately **not** on this list — they exist only to build the proxy label below, and per the label-trap rule (`trend_direction` from `trend_pct` in the flyrank-data skill) whatever a label is computed from can never also be a feature.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [26]:
# Named limitation, verified: not every client has March 2026 history at all.
coverage = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients,
        SUM(CASE WHEN gsc_data_start > DATE '2026-03-31' OR gsc_data_start IS NULL THEN 1 ELSE 0 END) AS no_march_coverage
    FROM {TABLES['dim_clients']}
""").df()

coverage


,n_clients,no_march_coverage
0,104,47.0


**Named limitation — unbalanced panel means March is not a level playing field across clients.** `dim_clients.gsc_data_start` shows some clients' tracking starts *after* 2026-03-31, so those clients contribute zero rows to `month=2026-03` — that's "not yet tracked," not "zero visibility." Any client-level comparison built on this March slice (e.g. "which clients look worst this month") has to check `gsc_data_start` first, or it will quietly read absence-of-history as decline. This is also why the real Lane 2 label can't be built from one calendar month alone: a future-window label (prior 90 days → next 30 days) needs each client to actually have prior-90-day history, which this single-month slice can't confirm on its own.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.